In [1]:
# !wget https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

In [3]:
# # Download GloVe (one time, ~1 min)
# ! wget http://nlp.stanford.edu/data/glove.6B.zip
# ! unzip glove.6B.zip


In [4]:
"""
SQuAD Answer Generation with GloVe Embeddings + Q/K Hypothesis Testing

EXPECTED PERFORMANCE:
- With GloVe embeddings: 40-55% F1 ✓
- Training time: ~40-50 minutes
- Can reach 50%+ with Q/K hypothesis
"""

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from transformers import GPT2Tokenizer
import json
from collections import Counter
import string
import re
from tqdm import tqdm
import numpy as np
import os
import urllib.request
import zipfile

# Configuration
TEST_QK_HYPOTHESIS = True  # Set True after baseline completes
QK_LR_MULTIPLIER = 20  # Q/K learn 2.5x faster

# Optimized for GloVe embeddings
D_MODEL = 300  # Match GloVe dimension exactly
N_HEADS = 6
N_LAYERS = 6
D_FF = 1200
MAX_SEQ_LEN = 256
MAX_ANSWER_LEN = 50
DROPOUT = 0.2
BATCH_SIZE = 32
ACCUMULATION_STEPS = 2  # Effective batch: 48
BASE_LR = 5e-5
WARMUP_STEPS = 1000
NUM_EPOCHS = 100
GRAD_CLIP = 0.5
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
TRAIN_SUBSET_SIZE = 60000  # More data with GloVe
VAL_SUBSET_SIZE = 10000


def download_and_extract_glove():
    """Download and extract GloVe embeddings"""
    glove_file = 'glove.6B.300d.txt'
    
    if os.path.exists(glove_file):
        print(f"✓ GloVe embeddings found: {glove_file}")
        return glove_file
    
    print("\n" + "="*70)
    print("DOWNLOADING GLOVE EMBEDDINGS")
    print("="*70)
    
    zip_file = 'glove.6B.zip'
    
    if not os.path.exists(zip_file):
        print("Downloading GloVe 6B (822MB)... This may take a few minutes")
        url = 'https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip'
        
        try:
            # Download with progress bar
            response = urllib.request.urlopen(url)
            total_size = int(response.headers.get('content-length', 0))
            
            with open(zip_file, 'wb') as f, tqdm(
                total=total_size, unit='B', unit_scale=True, desc='Downloading'
            ) as pbar:
                while True:
                    chunk = response.read(8192)
                    if not chunk:
                        break
                    f.write(chunk)
                    pbar.update(len(chunk))
            
            print("✓ Download complete!")
        except Exception as e:
            print(f"Download failed: {e}")
            print("\nAlternative: Download manually from:")
            print("  https://nlp.stanford.edu/projects/glove/")
            print("  or https://huggingface.co/stanfordnlp/glove")
            return None
    
    # Extract
    if os.path.exists(zip_file):
        print("Extracting GloVe embeddings...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            # Only extract the 300d file we need
            zip_ref.extract('glove.6B.300d.txt')
        print("✓ Extraction complete!")
        
        # Optionally remove zip to save space
        # os.remove(zip_file)
    
    if os.path.exists(glove_file):
        return glove_file
    else:
        print("⚠ GloVe file not found after extraction")
        return None


def load_glove_embeddings(glove_file, tokenizer, embedding_dim=300):
    """Load GloVe and create embedding matrix for GPT-2 tokenizer"""
    print("\n" + "="*70)
    print("LOADING GLOVE EMBEDDINGS")
    print("="*70)
    
    # Load GloVe vectors
    print("Reading GloVe file (this takes ~1 minute)...")
    glove_vectors = {}
    
    with open(glove_file, 'r', encoding='utf-8') as f:
        for line in tqdm(f, total=400000, desc="Loading GloVe"):
            values = line.rstrip().split(' ')
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_vectors[word] = vector
    
    print(f"✓ Loaded {len(glove_vectors):,} GloVe vectors")
    
    # Create embedding matrix for tokenizer vocabulary
    vocab_size = tokenizer.vocab_size
    embedding_matrix = np.random.normal(0, 0.1, (vocab_size, embedding_dim)).astype('float32')
    
    # Match tokenizer vocab with GloVe
    print("Matching tokenizer vocabulary with GloVe...")
    matched = 0
    
    for token, idx in tqdm(tokenizer.get_vocab().items(), desc="Matching"):
        # Try different matching strategies
        token_clean = token.replace('Ġ', '').replace('Ċ', '').lower().strip()
        
        if token in glove_vectors:
            embedding_matrix[idx] = glove_vectors[token]
            matched += 1
        elif token.lower() in glove_vectors:
            embedding_matrix[idx] = glove_vectors[token.lower()]
            matched += 1
        elif token_clean in glove_vectors:
            embedding_matrix[idx] = glove_vectors[token_clean]
            matched += 1
        # For subword tokens, try averaging character embeddings
        elif len(token_clean) > 0 and all(c.isalpha() for c in token_clean):
            # Use random but consistent embedding for unknown tokens
            pass
    
    match_rate = 100 * matched / vocab_size
    print(f"✓ Matched {matched:,}/{vocab_size:,} tokens ({match_rate:.1f}%)")
    print("="*70 + "\n")
    
    return torch.FloatTensor(embedding_matrix)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.last_attention_weights = None
        
    def forward(self, q, k, v, mask=None, save_attention=False):
        bs = q.size(0)
        
        q = self.q_linear(q).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k_linear(k).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        v = self.v_linear(v).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = torch.softmax(scores, dim=-1)
        if save_attention:
            self.last_attention_weights = attn.detach()
        
        attn = self.dropout(attn)
        context = torch.matmul(attn, v)
        context = context.transpose(1, 2).contiguous().view(bs, -1, self.n_heads * self.d_k)
        
        return self.out(context)


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
    def forward(self, x, mask=None, save_attention=False):
        # Pre-norm
        attn_out = self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), mask, save_attention)
        x = x + attn_out
        x = x + self.ff(self.norm2(x))
        return x


class GPTAnswerGenerator(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_seq_len, dropout=0.1, pretrained_embeddings=None):
        super().__init__()
        
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        
        # Initialize with pretrained embeddings if provided
        if pretrained_embeddings is not None:
            print("Initializing token embeddings with GloVe...")
            self.token_embedding.weight.data.copy_(pretrained_embeddings)
            print("✓ Token embeddings initialized with GloVe")
        
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.output = nn.Linear(d_model, vocab_size)
        
        # Weight tying
        self.output.weight = self.token_embedding.weight
        
        # Initialize non-embedding weights
        self._init_weights()
        
    def _init_weights(self):
        # Don't reinitialize token_embedding if using GloVe
        for name, p in self.named_parameters():
            if 'token_embedding' not in name and p.dim() > 1:
                nn.init.xavier_uniform_(p, gain=1/np.sqrt(2))
        
    def forward(self, x, mask=None, save_attention=False):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = self.token_embedding(x) + self.position_embedding(pos)
        x = self.emb_dropout(x)
        
        for layer in self.layers:
            x = layer(x, mask, save_attention)
        
        return self.output(self.norm(x))
    
    def get_attention_weights(self):
        return [layer.self_attn.last_attention_weights for layer in self.layers]


class SQuADDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_len, max_ans_len):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.max_ans_len = max_ans_len
        self.data = []
        
        with open(data_path, 'r') as f:
            squad = json.load(f)
        
        for article in squad['data']:
            for para in article['paragraphs']:
                ctx = para['context']
                for qa in para['qas']:
                    if not qa['is_impossible'] and qa['answers']:
                        ans = qa['answers'][0]['text']
                        ans_start = qa['answers'][0]['answer_start']
                        
                        # Extract relevant context window
                        start = max(0, ans_start - 200)
                        end = min(len(ctx), ans_start + len(ans) + 200)
                        focused_ctx = ctx[start:end]
                        
                        self.data.append({
                            'context': focused_ctx,
                            'question': qa['question'],
                            'answer': ans
                        })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Format: Q: question C: context A: answer
        prefix = f"Q: {item['question']} C: {item['context']} A:"
        answer = f" {item['answer']}"
        
        prefix_ids = self.tokenizer.encode(prefix, max_length=self.max_len-self.max_ans_len-2, 
                                          truncation=True, add_special_tokens=False)
        answer_ids = self.tokenizer.encode(answer, max_length=self.max_ans_len, 
                                          truncation=True, add_special_tokens=False)
        answer_ids.append(self.tokenizer.eos_token_id)
        
        input_ids = prefix_ids + answer_ids
        labels = [-100] * len(prefix_ids) + answer_ids
        
        # Pad
        while len(input_ids) < self.max_len:
            input_ids.append(self.tokenizer.pad_token_id)
            labels.append(-100)
        
        return {
            'input_ids': torch.tensor(input_ids[:self.max_len]),
            'labels': torch.tensor(labels[:self.max_len])
        }


def create_mask(seq_len, device):
    return (torch.triu(torch.ones(seq_len, seq_len, device=device), 1) == 0).unsqueeze(0).unsqueeze(0)


def normalize_answer(s):
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(c for c in s if c not in string.punctuation)
    return ' '.join(s.split())


def f1_score(pred, truth):
    pred_tok = normalize_answer(pred).split()
    truth_tok = normalize_answer(truth).split()
    
    if not pred_tok or not truth_tok:
        return int(pred_tok == truth_tok)
    
    common = Counter(pred_tok) & Counter(truth_tok)
    if not common:
        return 0
    
    prec = sum(common.values()) / len(pred_tok)
    rec = sum(common.values()) / len(truth_tok)
    return 2 * prec * rec / (prec + rec)


def exact_match(pred, truth):
    return int(normalize_answer(pred) == normalize_answer(truth))


def train_epoch(model, loader, opt, sched, device, epoch):
    model.train()
    total_loss = 0
    opt.zero_grad()
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}")
    for i, batch in enumerate(pbar):
        inp = batch['input_ids'].to(device)
        lbl = batch['labels'].to(device)
        
        mask = create_mask(inp.size(1), device)
        logits = model(inp, mask)
        
        # Shift for next-token prediction
        loss = nn.functional.cross_entropy(
            logits[:, :-1].reshape(-1, logits.size(-1)),
            lbl[:, 1:].reshape(-1),
            ignore_index=-100,
            label_smoothing=LABEL_SMOOTHING
        )
        
        loss = loss / ACCUMULATION_STEPS
        loss.backward()
        
        if (i + 1) % ACCUMULATION_STEPS == 0:
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            sched.step()
            opt.zero_grad()
        
        total_loss += loss.item() * ACCUMULATION_STEPS
        pbar.set_postfix({'loss': f'{loss.item() * ACCUMULATION_STEPS:.3f}'})
    
    return total_loss / len(loader)


def generate(model, tokenizer, context, question, device, max_len=50):
    model.eval()
    
    prompt = f"Q: {question} C: {context} A:"
    ids = tokenizer.encode(prompt, max_length=MAX_SEQ_LEN-max_len-5, 
                          truncation=True, add_special_tokens=False, return_tensors='pt').to(device)
    
    start_len = ids.size(1)
    
    with torch.no_grad():
        for _ in range(max_len):
            if ids.size(1) >= MAX_SEQ_LEN:
                break
            
            mask = create_mask(ids.size(1), device)
            logits = model(ids, mask)
            next_tok = logits[:, -1].argmax(-1, keepdim=True)
            ids = torch.cat([ids, next_tok], 1)
            
            if next_tok.item() == tokenizer.eos_token_id:
                break
    
    return tokenizer.decode(ids[0, start_len:], skip_special_tokens=True).strip()


def evaluate(model, dataset, tokenizer, device, n_samples=300):
    model.eval()
    f1_sum = em_sum = 0
    
    if isinstance(dataset, Subset):
        items = [dataset.dataset.data[dataset.indices[i]] for i in range(min(n_samples, len(dataset)))]
    else:
        items = dataset.data[:n_samples]
    
    for item in tqdm(items, desc="Eval"):
        pred = generate(model, tokenizer, item['context'], item['question'], device)
        f1_sum += f1_score(pred, item['answer'])
        em_sum += exact_match(pred, item['answer'])
    
    return {'f1': f1_sum / len(items), 'em': em_sum / len(items)}


def analyze_attention(model, dataset, tokenizer, device, n=30):
    model.eval()
    scores = []
    
    if isinstance(dataset, Subset):
        items = [dataset.dataset.data[dataset.indices[i]] for i in range(min(n, len(dataset)))]
    else:
        items = dataset.data[:n]
    
    for item in items:
        prompt = f"Q: {item['question']} C: {item['context']} A:"
        ids = tokenizer.encode(prompt, max_length=MAX_SEQ_LEN-MAX_ANSWER_LEN, 
                              truncation=True, add_special_tokens=False, return_tensors='pt').to(device)
        
        with torch.no_grad():
            mask = create_mask(ids.size(1), device)
            model(ids, mask, save_attention=True)
            
            weights = model.get_attention_weights()
            if weights[0] is not None:
                avg = torch.stack([w[0] for w in weights if w is not None]).mean(0)
                scores.append(avg[0].mean().item())
    
    return np.mean(scores) if scores else 0

In [4]:
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print("="*70)
    print("SQUAD ANSWER GENERATION WITH GLOVE EMBEDDINGS")
    print("="*70)
    #print(f"Expected F1: 40-55% (with GloVe)")
    print(f"Model: {N_LAYERS}L, {D_MODEL}d, {N_HEADS}h")
    print(f"Device: {device}")
    print("="*70 + "\n")
    
    # Download and load GloVe
    glove_file = download_and_extract_glove()
    
    if glove_file is None:
        print("\n WARNING: Could not load GloVe embeddings")
        print("Proceeding without pretrained embeddings (expect 15-25% F1)")
        pretrained_embeddings = None
    
    # Download SQuAD datasets
    for name in ['train-v2.0.json', 'dev-v2.0.json']:
        if not os.path.exists(name):
            print(f"Downloading {name}...")
            urllib.request.urlretrieve(
                f'https://rajpurkar.github.io/SQuAD-explorer/dataset/{name}', name)
    
    # Setup tokenizer
    print("Loading tokenizer...")
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    
    # Load GloVe embeddings for tokenizer
    if glove_file:
        pretrained_embeddings = load_glove_embeddings(glove_file, tokenizer, D_MODEL)
    else:
        pretrained_embeddings = None
    
    # Load datasets
    print("Loading datasets...")
    full_train = SQuADDataset('train-v2.0.json', tokenizer, MAX_SEQ_LEN, MAX_ANSWER_LEN)
    full_val = SQuADDataset('dev-v2.0.json', tokenizer, MAX_SEQ_LEN, MAX_ANSWER_LEN)
    
    train_ds = Subset(full_train, torch.randperm(len(full_train))[:TRAIN_SUBSET_SIZE])
    val_ds = Subset(full_val, torch.randperm(len(full_val))[:VAL_SUBSET_SIZE])
    
    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}\n")
    
    loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    
    # Model
    print("Initializing model...")
    n_seed = [1234,1235,1236,1237,1238]
    for seed_ in n_seed:
        print("Training for seed", seed_)
        torch.manual_seed(seed_)
        model = GPTAnswerGenerator(
            vocab_size=tokenizer.vocab_size,
            d_model=D_MODEL,
            n_heads=N_HEADS,
            n_layers=N_LAYERS,
            d_ff=D_FF,
            max_seq_len=MAX_SEQ_LEN,
            dropout=DROPOUT,
            pretrained_embeddings=pretrained_embeddings
        ).to(device)

        total_params = sum(p.numel() for p in model.parameters()) / 1e6
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
        print(f"Total parameters: {total_params:.1f}M")
        print(f"Trainable parameters: {trainable_params:.1f}M\n")

        # Optimizer with differential learning rates for embeddings
        if TEST_QK_HYPOTHESIS:
            print("="*70)
            print(f"TESTING Q/K HYPOTHESIS - Q/K LR = {QK_LR_MULTIPLIER}x")
            print("="*70 + "\n")
            embedding_params = [model.token_embedding.weight]
            qk = [p for n, p in model.named_parameters() if 'q_linear' in n or 'k_linear' in n]
            other = [p for n, p in model.named_parameters() if 'q_linear' not in n and 
                     'k_linear' not in n and 'token_embedding' not in n]

            print(f"Q/K params: {sum(p.numel() for p in qk)/1e6:.1f}M")
            print(f"Other params: {sum(p.numel() for p in other)/1e6:.1f}M\n")

            opt = torch.optim.AdamW([
                {'params':embedding_params,'lr': BASE_LR * 0.1, 'weight_decay': 0},
                {'params': qk, 'lr': BASE_LR * QK_LR_MULTIPLIER, 'weight_decay': WEIGHT_DECAY},
                {'params': other, 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY}
            ])

            sched = torch.optim.lr_scheduler.OneCycleLR(
                opt, [BASE_LR * 0.1,BASE_LR * QK_LR_MULTIPLIER, BASE_LR],
                total_steps=len(loader) * NUM_EPOCHS,
                pct_start=WARMUP_STEPS / (len(loader) * NUM_EPOCHS)
            )
        else:
            print("="*70)
            print("BASELINE (Standard LR)")
            print("="*70 + "\n")

            # Use lower LR for pretrained embeddings if they exist
            if pretrained_embeddings is not None:
                embedding_params = [model.token_embedding.weight]
                other_params = [p for n, p in model.named_parameters() if 'token_embedding' not in n]

                opt = torch.optim.AdamW([
                    {'params': embedding_params, 'lr': BASE_LR * 0.1, 'weight_decay': 0},  # Fine-tune slowly
                    {'params': other_params, 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY}
                ])

                print("Using differential LR: embeddings=0.1x, other=1.0x\n")
            else:
                opt = torch.optim.AdamW(model.parameters(), BASE_LR, weight_decay=WEIGHT_DECAY)

            sched = torch.optim.lr_scheduler.OneCycleLR(
                opt,
                max_lr=BASE_LR if pretrained_embeddings is None else [BASE_LR * 0.1, BASE_LR],
                total_steps=len(loader) * NUM_EPOCHS,
                pct_start=WARMUP_STEPS / (len(loader) * NUM_EPOCHS)
            )

        # Train
        best_f1 = 0
        results = {'loss': [], 'train_f1': [], 'val_f1': [], 'val_em': [], 'attn': []}

        for e in range(NUM_EPOCHS):
            print(f"\n{'='*70}")
            print(f"EPOCH {e+1}/{NUM_EPOCHS}")
            print('='*70)

            loss = train_epoch(model, loader, opt, sched, device, e+1)
            results['loss'].append(loss)
            print(f"\nLoss: {loss:.4f}")

            # Eval
            train_m = evaluate(model, train_ds, tokenizer, device, 200)
            val_m = evaluate(model, val_ds, tokenizer, device, 300)

            results['train_f1'].append(train_m['f1'])
            results['val_f1'].append(val_m['f1'])
            results['val_em'].append(val_m['em'])

            gap = train_m['f1'] - val_m['f1']
            print(f"Train F1: {train_m['f1']:.4f} | Val F1: {val_m['f1']:.4f} | Gap: {gap:.4f} | EM: {val_m['em']:.4f}")

            # Sample
            if e % 2 == 0:
                item = val_ds.dataset.data[val_ds.indices[0]]
                pred = generate(model, tokenizer, item['context'], item['question'], device)
                print(f"\nSample:")
                print(f"  Q: {item['question'][:60]}...")
                print(f"  True: {item['answer']}")
                print(f"  Pred: {pred}")
                print(f"  F1: {f1_score(pred, item['answer']):.3f}")

            # Attention
            if e % 4 == 0 and TEST_QK_HYPOTHESIS:
                attn = analyze_attention(model, val_ds, tokenizer, device)
                results['attn'].append(attn)
                print(f"Attention: {attn:.4f}")

            # Save best
            if val_m['f1'] > best_f1:
                best_f1 = val_m['f1']
                name = 'best_qk_'+str(seed_)+'.pt' if TEST_QK_HYPOTHESIS else 'best_baseline_'+str(seed_)+'.pt'
                torch.save({'model': model.state_dict(), 'f1': best_f1, 'epoch': e+1}, name)
                print(f"✓ SAVED! Best F1: {best_f1:.4f}")

        # Final
        print(f"\n{'='*70}")
        print("FINAL RESULTS")
        print('='*70)
        print(f"Best Val F1: {best_f1*100:.1f}%")
        print(f"Final Val F1: {results['val_f1'][-1]*100:.1f}%")
        print(f"Final EM: {results['val_em'][-1]*100:.1f}%")
        print(f"Train-Val Gap: {results['train_f1'][-1] - results['val_f1'][-1]:.4f}")

SQUAD ANSWER GENERATION WITH GLOVE EMBEDDINGS
Model: 6L, 300d, 6h
Device: cuda

✓ GloVe embeddings found: glove.6B.300d.txt
Loading tokenizer...

LOADING GLOVE EMBEDDINGS
Reading GloVe file (this takes ~1 minute)...


Loading GloVe: 100%|█████████████████| 400000/400000 [00:19<00:00, 20156.51it/s]


✓ Loaded 400,000 GloVe vectors
Matching tokenizer vocabulary with GloVe...


Matching: 100%|███████████████████████| 50257/50257 [00:00<00:00, 368110.81it/s]

✓ Matched 43,058/50,257 tokens (85.7%)



...

Train F1: 0.7651 | Val F1: 0.6107 | Gap: 0.1544 | EM: 0.3833

Sample:
  Q: How many tons of carbon are absorbed the Amazon in a typical...
  True: 1.5 gigatons
  Pred: 1.5
  F1: 0.667

EPOCH 100/100


Epoch 100: 100%|████████████████| 1875/1875 [05:02<00:00,  6.19it/s, loss=2.596]



Loss: 2.3680


Eval: 100%|███████████████████████████████████| 300/300 [00:10<00:00, 27.45it/s]

Train F1: 0.7564 | Val F1: 0.5942 | Gap: 0.1622 | EM: 0.3733

FINAL RESULTS
Best Val F1: 61.1%
Final Val F1: 59.4%
Final EM: 37.3%
Train-Val Gap: 0.1622


In [5]:
# Eval
train_m = evaluate(model, train_ds, tokenizer, device, 40000)

Eval: 100%|███████████████████████████████| 40000/40000 [20:52<00:00, 31.94it/s]


In [ ]:
train_m["f1"],train_m["em"]

In [ ]:
val_m = evaluate(model, val_ds, tokenizer, device, 20000)

In [ ]:
val_m["f1"],val_m["em"]

In [10]:
# Add this if not present
from transformers import GPT2TokenizerFast

# loading and evaluating faster_qk model on train data for f1 score


seeds_list = [1234, 1235,1236, 1237, 1238]



print("Loading tokenizer...")
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
DROPOUT = 0.2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

glove_file = download_and_extract_glove()
pretrained_embeddings = load_glove_embeddings(glove_file, tokenizer, D_MODEL)



# Load datasets
print("Loading datasets...")
full_train = SQuADDataset('train-v2.0.json', tokenizer, MAX_SEQ_LEN, MAX_ANSWER_LEN)


train_ds = Subset(full_train, torch.randperm(len(full_train))[:40000])



for nseed in seeds_list:
    BASELINE_CKPT = """/home/cs22d010/Squad_QA/Squad_Results/Rebuiding-From-Scratch/best_qk_"""+str(nseed)+""".pt""" #"best_qk.pt"  #"best_baseline.pt"  #  "best_qk.pt"
    model_kwargs = dict(vocab_size=tokenizer.vocab_size,
                        d_model=D_MODEL, n_heads=N_HEADS,
                        n_layers=N_LAYERS, d_ff=D_FF,
                        max_seq_len=MAX_SEQ_LEN, 
                        dropout=DROPOUT,pretrained_embeddings=pretrained_embeddings)
    print("Initializing baseline model instance..."+str(nseed))
    baseline = GPTAnswerGenerator(**model_kwargs).to(device)
    baseline_ckpt = torch.load(BASELINE_CKPT, map_location=device)
    baseline.load_state_dict(baseline_ckpt["model"])
    print("Baseline model loaded.")
    
    train_m = evaluate(baseline, train_ds, tokenizer, device,40000)

    print(train_m)

Loading tokenizer...
✓ GloVe embeddings found: glove.6B.300d.txt

LOADING GLOVE EMBEDDINGS
Reading GloVe file (this takes ~1 minute)...


Loading GloVe: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400000/400000 [00:32<00:00, 12352.13it/s]


✓ Loaded 400,000 GloVe vectors
Matching tokenizer vocabulary with GloVe...


Matching: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50257/50257 [00:00<00:00, 309838.63it/s]

✓ Matched 43,058/50,257 tokens (85.7%)



...

{'f1': 0.7313670387150584, 'em': 0.57665}
Initializing baseline model instance...1237
Initializing token embeddings with GloVe...
✓ Token embeddings initialized with GloVe
Baseline model loaded.


Eval: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40000/40000 [23:28<00:00, 28.39it/s]


{'f1': 0.718852340662438, 'em': 0.571575}
Initializing baseline model instance...1238
Initializing token embeddings with GloVe...
✓ Token embeddings initialized with GloVe
Baseline model loaded.


Eval: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40000/40000 [26:30<00:00, 25.14it/s]

{'f1': 0.733321006810185, 'em': 0.57715}


In [11]:

# loading and evaluating baseline model on train data for f1 score



seeds_list = [1234, 1235,1236, 1237, 1238]



print("Loading tokenizer...")
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
DROPOUT = 0.2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

glove_file = download_and_extract_glove()
pretrained_embeddings = load_glove_embeddings(glove_file, tokenizer, D_MODEL)



# Load datasets
print("Loading datasets...")
full_train = SQuADDataset('train-v2.0.json', tokenizer, MAX_SEQ_LEN, MAX_ANSWER_LEN)


train_ds = Subset(full_train, torch.randperm(len(full_train))[:40000])



for nseed in seeds_list:
    BASELINE_CKPT = """/home/cs22d010/Squad_QA/Squad_Results/Rebuiding-From-Scratch/best_baseline_"""+str(nseed)+""".pt""" #"best_qk.pt"  #"best_baseline.pt"  #  "best_qk.pt"
    model_kwargs = dict(vocab_size=tokenizer.vocab_size,
                        d_model=D_MODEL, n_heads=N_HEADS,
                        n_layers=N_LAYERS, d_ff=D_FF,
                        max_seq_len=MAX_SEQ_LEN, 
                        dropout=DROPOUT,pretrained_embeddings=pretrained_embeddings)
    print("Initializing baseline model instance..."+str(nseed))
    baseline = GPTAnswerGenerator(**model_kwargs).to(device)
    baseline_ckpt = torch.load(BASELINE_CKPT, map_location=device)
    baseline.load_state_dict(baseline_ckpt["model"])
    print("Baseline model loaded.")
    
    train_m = evaluate(baseline, train_ds, tokenizer, device,40000)

    print(train_m)

Loading tokenizer...
✓ GloVe embeddings found: glove.6B.300d.txt

LOADING GLOVE EMBEDDINGS
Reading GloVe file (this takes ~1 minute)...


Loading GloVe: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400000/400000 [00:22<00:00, 17979.49it/s]


✓ Loaded 400,000 GloVe vectors
Matching tokenizer vocabulary with GloVe...


Matching: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50257/50257 [00:00<00:00, 330287.53it/s]

✓ Matched 43,058/50,257 tokens (85.7%)

Loading datasets...


...

{'f1': 0.736969098824084, 'em': 0.585725}
Initializing baseline model instance...1237
Initializing token embeddings with GloVe...
✓ Token embeddings initialized with GloVe
Baseline model loaded.


Eval: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40000/40000 [26:17<00:00, 25.35it/s]


{'f1': 0.8185366758991849, 'em': 0.702075}
Initializing baseline model instance...1238
Initializing token embeddings with GloVe...
✓ Token embeddings initialized with GloVe
Baseline model loaded.


Eval: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40000/40000 [22:56<00:00, 29.06it/s]

{'f1': 0.8426166410023589, 'em': 0.738275}


In [12]:
# Add this if not present
from transformers import GPT2TokenizerFast

# loading and evaluating faster_qk model on train data for f1 score


seeds_list = [1234, 1235,1236, 1237, 1238]



print("Loading tokenizer...")
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
DROPOUT = 0.2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

glove_file = download_and_extract_glove()
pretrained_embeddings = load_glove_embeddings(glove_file, tokenizer, D_MODEL)



# Load datasets
print("Loading datasets...")
full_train = SQuADDataset('dev-v2.0.json', tokenizer, MAX_SEQ_LEN, MAX_ANSWER_LEN)


train_ds = Subset(full_train, torch.randperm(len(full_train))[:5000])



for nseed in seeds_list:
    BASELINE_CKPT = """/home/cs22d010/Squad_QA/Squad_Results/Rebuiding-From-Scratch/best_qk_"""+str(nseed)+""".pt""" #"best_qk.pt"  #"best_baseline.pt"  #  "best_qk.pt"
    model_kwargs = dict(vocab_size=tokenizer.vocab_size,
                        d_model=D_MODEL, n_heads=N_HEADS,
                        n_layers=N_LAYERS, d_ff=D_FF,
                        max_seq_len=MAX_SEQ_LEN, 
                        dropout=DROPOUT,pretrained_embeddings=pretrained_embeddings)
    print("Initializing baseline model instance..."+str(nseed))
    baseline = GPTAnswerGenerator(**model_kwargs).to(device)
    baseline_ckpt = torch.load(BASELINE_CKPT, map_location=device)
    baseline.load_state_dict(baseline_ckpt["model"])
    print("Baseline model loaded.")
    
    train_m = evaluate(baseline, train_ds, tokenizer, device,40000)

    print(train_m)

Loading tokenizer...
✓ GloVe embeddings found: glove.6B.300d.txt

LOADING GLOVE EMBEDDINGS
Reading GloVe file (this takes ~1 minute)...


Loading GloVe: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400000/400000 [00:21<00:00, 18850.05it/s]


✓ Loaded 400,000 GloVe vectors
Matching tokenizer vocabulary with GloVe...


Matching: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50257/50257 [00:00<00:00, 380099.89it/s]

✓ Matched 43,058/50,257 tokens (85.7%)



...

{'f1': 0.6024746737280913, 'em': 0.4022}
Initializing baseline model instance...1237
Initializing token embeddings with GloVe...
✓ Token embeddings initialized with GloVe
Baseline model loaded.


Eval: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5000/5000 [02:44<00:00, 30.34it/s]


{'f1': 0.5653474290713689, 'em': 0.3666}
Initializing baseline model instance...1238
Initializing token embeddings with GloVe...
✓ Token embeddings initialized with GloVe
Baseline model loaded.


Eval: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5000/5000 [03:14<00:00, 25.76it/s]

{'f1': 0.6014094317562373, 'em': 0.3922}


In [13]:

# loading and evaluating baseline model on train data for f1 score



seeds_list = [1234, 1235,1236, 1237, 1238]



print("Loading tokenizer...")
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
DROPOUT = 0.2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

glove_file = download_and_extract_glove()
pretrained_embeddings = load_glove_embeddings(glove_file, tokenizer, D_MODEL)



# Load datasets
print("Loading datasets...")
full_train = SQuADDataset('dev-v2.0.json', tokenizer, MAX_SEQ_LEN, MAX_ANSWER_LEN)


train_ds = Subset(full_train, torch.randperm(len(full_train))[:5000])



for nseed in seeds_list:
    BASELINE_CKPT = """/home/cs22d010/Squad_QA/Squad_Results/Rebuiding-From-Scratch/best_baseline_"""+str(nseed)+""".pt""" #"best_qk.pt"  #"best_baseline.pt"  #  "best_qk.pt"
    model_kwargs = dict(vocab_size=tokenizer.vocab_size,
                        d_model=D_MODEL, n_heads=N_HEADS,
                        n_layers=N_LAYERS, d_ff=D_FF,
                        max_seq_len=MAX_SEQ_LEN, 
                        dropout=DROPOUT,pretrained_embeddings=pretrained_embeddings)
    print("Initializing baseline model instance..."+str(nseed))
    baseline = GPTAnswerGenerator(**model_kwargs).to(device)
    baseline_ckpt = torch.load(BASELINE_CKPT, map_location=device)
    baseline.load_state_dict(baseline_ckpt["model"])
    print("Baseline model loaded.")
    
    train_m = evaluate(baseline, train_ds, tokenizer, device,40000)

    print(train_m)

Loading tokenizer...
✓ GloVe embeddings found: glove.6B.300d.txt

LOADING GLOVE EMBEDDINGS
Reading GloVe file (this takes ~1 minute)...


Loading GloVe: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400000/400000 [00:21<00:00, 18715.21it/s]


✓ Loaded 400,000 GloVe vectors
Matching tokenizer vocabulary with GloVe...


Matching: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50257/50257 [00:00<00:00, 339416.85it/s]

✓ Matched 43,058/50,257 tokens (85.7%)



...

{'f1': 0.553259539025865, 'em': 0.3524}
Initializing baseline model instance...1237
Initializing token embeddings with GloVe...
✓ Token embeddings initialized with GloVe
Baseline model loaded.


Eval: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5000/5000 [03:25<00:00, 24.37it/s]


{'f1': 0.571212736113227, 'em': 0.367}
Initializing baseline model instance...1238
Initializing token embeddings with GloVe...
✓ Token embeddings initialized with GloVe
Baseline model loaded.


Eval: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5000/5000 [02:55<00:00, 28.44it/s]

{'f1': 0.5921552339345275, 'em': 0.3914}
